In [6]:
import glob
import os
import stim
import numpy as np
import pickle
from tqdm import tqdm
from css import compute_css_logical_operators
from css_simulator import construct_css_resource_state
import time
# # ── Error rate / shot configuration ──────────────────────────────────────────
# error_rates      = np.linspace(1.0, 0.0, 20, endpoint=False)
# shot_counts      = [3000 * (2 ** i) for i in range(20)]
# error_shots_dict = dict(zip(error_rates, shot_counts))

# # ── Collect all HGP code files ────────────────────────────────────────────────
# code_dir   = "/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/ECC_gen/code_lib/hgp_code_lib/"
# code_files = sorted(glob.glob(code_dir + "hgp_*.pkl"))

# tqdm.write(f"{'='*60}")
# tqdm.write(f"Found {len(code_files)} code files to process.")
# tqdm.write(f"Error rates : {len(error_shots_dict)} levels from {max(error_rates):.3f} → {min(error_rates):.3f}")
# tqdm.write(f"Shot counts : {min(shot_counts):,} → {max(shot_counts):,}")
# tqdm.write(f"{'='*60}\n")

# # ── Outer loop: codes ─────────────────────────────────────────────────────────
# for code_idx, code_path in enumerate(tqdm(code_files, desc="Codes", position=0, leave=True), start=1):

#     code_start_time        = time.time()
#     base                   = os.path.basename(code_path)
#     _, n_str, k_str, d_str = base.replace(".pkl", "").split("_")
#     n_code, k_code, d_code = int(n_str), int(k_str), int(d_str)
#     tag                    = f"{n_code}_{k_code}_{d_code}"

#     tqdm.write(f"\n[Code {code_idx}/{len(code_files)}] {base}")
#     tqdm.write(f"  Parameters : [[n={n_code}, k={k_code}, d={d_code}]]")
#     tqdm.write(f"  Rate       : {k_code/n_code:.4f}")

#     # ── Load matrices ─────────────────────────────────────────────────────────
#     tqdm.write(f"  Loading matrices ...")
#     with open(code_path, "rb") as f:
#         code = pickle.load(f)

#     hgp_x = code["hgp_x"].astype(np.uint8, copy=False)
#     hgp_z = code["hgp_z"].astype(np.uint8, copy=False)
#     tqdm.write(f"  Hx shape   : {hgp_x.shape}  |  Hz shape: {hgp_z.shape}")

#     # ── Compute logical operators ─────────────────────────────────────────────
#     tqdm.write(f"  Computing logical operators ...")
#     logical_x_matrix, logical_z_matrix = compute_css_logical_operators(hgp_x, hgp_z)
#     num_logical_qubits, num_physical_qubits = logical_x_matrix.shape
#     num_total_qubits = num_physical_qubits + num_logical_qubits
#     tqdm.write(f"  Physical qubits : {num_physical_qubits}")
#     tqdm.write(f"  Logical  qubits : {num_logical_qubits}")
#     tqdm.write(f"  Total    qubits : {num_total_qubits}")

#     # ── Save logical operators ────────────────────────────────────────────────
#     operators_path = f"operators_{tag}.pkl"
#     tqdm.write(f"  Saving logical operators to {operators_path} ...")
#     with open(operators_path, "wb") as f:
#         pickle.dump({
#             "hgp_x":           hgp_x,
#             "hgp_z":           hgp_z,
#             "logical_x":       logical_x_matrix,
#             "logical_z":       logical_z_matrix,
#             "n":               n_code,
#             "k":               k_code,
#             "d_est":           d_code,
#         }, f)
#     tqdm.write(f"  Saved: {operators_path}")

#     # ── Build base circuit ────────────────────────────────────────────────────
#     tqdm.write(f"  Building base circuit ...")
#     circuit_start = time.time()
#     tableau       = construct_css_resource_state(
#         hgp_x.toarray(),
#         hgp_z.toarray(),
#         logical_x_matrix.toarray(),
#         logical_z_matrix.toarray()
#     )
#     circuit_init      = (tableau + tableau).to_circuit()
#     circuit_build_time = time.time() - circuit_start
#     tqdm.write(f"  Circuit built in {circuit_build_time:.2f}s : {len(circuit_init)} instructions")

#     # ── Save base circuit ─────────────────────────────────────────────────────
#     circuit_path = f"circuit_{tag}.pkl"
#     tqdm.write(f"  Saving base circuit to {circuit_path} ...")
#     with open(circuit_path, "wb") as f:
#         pickle.dump({
#             "circuit_init":        circuit_init,
#             "num_physical_qubits": num_physical_qubits,
#             "num_logical_qubits":  num_logical_qubits,
#             "num_total_qubits":    num_total_qubits,
#             "circuit_build_time":  circuit_build_time,
#             "n":                   n_code,
#             "k":                   k_code,
#             "d_est":               d_code,
#         }, f)
#     tqdm.write(f"  Saved: {circuit_path}")

#     # ── Inner loop: error rates ───────────────────────────────────────────────
#     tqdm.write(f"  Starting sampling over {len(error_shots_dict)} error rates ...")
#     results_dict       = {}
#     total_shots_so_far = 0

#     for depol_error_rate, num_shots in tqdm(
#         error_shots_dict.items(),
#         desc=f"  [[{n_code},{k_code},{d_code}]] error rates",
#         position=1,
#         leave=False
#     ):
#         shot_start = time.time()
#         circuit    = circuit_init.copy()
#         circuit.append("DEPOLARIZE1", list(range(num_physical_qubits)), depol_error_rate)

#         for qubit_index in range(num_total_qubits):
#             target_qubit = qubit_index + num_total_qubits
#             circuit.append("MXX", [qubit_index, target_qubit])
#             circuit.append("MZZ", [qubit_index, target_qubit])

#         tqdm.write(f"    Sampling : depol={depol_error_rate:.4f}  shots={num_shots:>12,} ...")
#         sampler       = circuit.compile_sampler()
#         measurements  = np.array(sampler.sample(shots=num_shots), dtype=np.uint8)
#         shot_duration = time.time() - shot_start
#         total_shots_so_far += num_shots

#         results_dict[depol_error_rate] = {
#             "num_shots":         num_shots,
#             "measurements":      measurements,
#             "sampling_time_sec": shot_duration,
#         }
#         tqdm.write(f"    Done     : depol={depol_error_rate:.4f}  shots={num_shots:>12,}  "
#                    f"shape={measurements.shape}  "
#                    f"time={shot_duration:.2f}s  "
#                    f"cumulative={total_shots_so_far:,}")

#     # ── Save samples ──────────────────────────────────────────────────────────
#     sample_path = f"sample_{tag}.pkl"
#     tqdm.write(f"  Saving samples to {sample_path} ...")
#     with open(sample_path, "wb") as f:
#         pickle.dump(results_dict, f)
#     tqdm.write(f"  Saved: {sample_path}")

#     # ── Save metadata ─────────────────────────────────────────────────────────
#     meta_path      = f"meta_{tag}.pkl"
#     code_duration  = time.time() - code_start_time
#     tqdm.write(f"  Saving metadata to {meta_path} ...")
#     with open(meta_path, "wb") as f:
#         pickle.dump({
#             "n":                    n_code,
#             "k":                    k_code,
#             "d_est":                d_code,
#             "rate":                 k_code / n_code,
#             "hgp_x_shape":          hgp_x.shape,
#             "hgp_z_shape":          hgp_z.shape,
#             "num_physical_qubits":  num_physical_qubits,
#             "num_logical_qubits":   num_logical_qubits,
#             "num_total_qubits":     num_total_qubits,
#             "error_rates":          list(error_shots_dict.keys()),
#             "shot_counts":          list(error_shots_dict.values()),
#             "total_shots":          total_shots_so_far,
#             "circuit_build_time":   circuit_build_time,
#             "total_time_sec":       code_duration,
#             "source_file":          code_path,
#             "timestamp":            time.strftime("%Y-%m-%d %H:%M:%S"),
#         }, f)
#     tqdm.write(f"  Saved    : {meta_path}")
#     tqdm.write(f"  Total time for this code : {code_duration:.2f}s")
#     tqdm.write(f"[Code {code_idx}/{len(code_files)}] COMPLETE ✓")

# tqdm.write(f"\n{'='*60}")
# tqdm.write(f"All {len(code_files)} codes processed successfully.")
# tqdm.write(f"{'='*60}")

In [7]:
# ── Error rate / shot configuration ──────────────────────────────────────────
error_rates      = np.linspace(1.0, 0.0, 20, endpoint=False)
shot_counts      = [3000 * (2 ** i) for i in range(20)]
error_shots_dict = dict(zip(error_rates, shot_counts))

# ── Collect all HGP code files ────────────────────────────────────────────────
code_dir   = "/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/ECC_gen/code_lib/hgp_code_lib2/"
code_files = sorted(glob.glob(code_dir + "hgp_*.pkl"))

tqdm.write(f"{'='*60}")
tqdm.write(f"Found {len(code_files)} code files.")
tqdm.write(f"{'='*60}\n")

Found 11 code files.



In [8]:
# tqdm.write("PHASE 1: Computing logical operators\n")

# for code_idx, code_path in enumerate(tqdm(code_files, desc="Logical Operators", position=0, leave=True), start=1):

#     base                   = os.path.basename(code_path)
#     _, n_str, k_str, d_str = base.replace(".pkl", "").split("_")
#     n_code, k_code, d_code = int(n_str), int(k_str), int(d_str)
#     tag                    = f"{n_code}_{k_code}_{d_code}"

#     tqdm.write(f"\n[{code_idx}/{len(code_files)}] {base}")

#     # Load matrices
#     with open(code_path, "rb") as f:
#         code = pickle.load(f)

#     hgp_x = code["hgp_x"].astype(np.uint8, copy=False)
#     hgp_z = code["hgp_z"].astype(np.uint8, copy=False)
#     tqdm.write(f"  Hx shape : {hgp_x.shape}  |  Hz shape : {hgp_z.shape}")

#     # Compute logical operators
#     tqdm.write(f"  Computing logical operators ...")
#     start            = time.time()
#     logical_x_matrix, logical_z_matrix = compute_css_logical_operators(hgp_x, hgp_z)
#     num_logical_qubits, num_physical_qubits = logical_x_matrix.shape
#     num_total_qubits = num_physical_qubits + num_logical_qubits
#     tqdm.write(f"  Done in {time.time() - start:.2f}s")
#     tqdm.write(f"  Physical : {num_physical_qubits}  |  Logical : {num_logical_qubits}  |  Total : {num_total_qubits}")

#     # Save
#     operators_path = f"operators_{tag}.pkl"
#     with open(operators_path, "wb") as f:
#         pickle.dump({
#             "hgp_x":                hgp_x,
#             "hgp_z":                hgp_z,
#             "logical_x":            logical_x_matrix,
#             "logical_z":            logical_z_matrix,
#             "num_physical_qubits":  num_physical_qubits,
#             "num_logical_qubits":   num_logical_qubits,
#             "num_total_qubits":     num_total_qubits,
#             "n":                    n_code,
#             "k":                    k_code,
#             "d_est":                d_code,
#         }, f)
#     tqdm.write(f"  Saved : {operators_path}")

# tqdm.write(f"\nPHASE 1 COMPLETE ✓ — {len(code_files)} operator files saved.\n")

In [9]:
# tqdm.write("PHASE 2: Building base circuits\n")

# operator_files = sorted(glob.glob("operators_*.pkl"))
# tqdm.write(f"Found {len(operator_files)} operator files.\n")

# for code_idx, op_path in enumerate(tqdm(operator_files, desc="Base Circuits", position=0, leave=True), start=1):

#     base                   = os.path.basename(op_path)
#     _, n_str, k_str, d_str = base.replace(".pkl", "").split("_")
#     n_code, k_code, d_code = int(n_str), int(k_str), int(d_str)
#     tag                    = f"{n_code}_{k_code}_{d_code}"

#     tqdm.write(tag)

#     # Load operators
#     with open(op_path, "rb") as f:
#         ops = pickle.load(f)

#     hgp_x              = ops["hgp_x"]
#     hgp_z              = ops["hgp_z"]
#     logical_x_matrix   = ops["logical_x"]
#     logical_z_matrix   = ops["logical_z"]
#     num_physical_qubits = ops["num_physical_qubits"]
#     num_logical_qubits  = ops["num_logical_qubits"]
#     num_total_qubits    = ops["num_total_qubits"]
#     tqdm.write(f"  Loaded operators for [[n={n_code}, k={k_code}, d={d_code}]]")

#     # Build circuit
#     tqdm.write(f"  Building base circuit ...")
#     start   = time.time()
#     tableau = construct_css_resource_state(
#         hgp_x.toarray(),
#         hgp_z.toarray(),
#         logical_x_matrix.toarray(),
#         logical_z_matrix.toarray()
#     )
#     circuit_init       = (tableau + tableau).to_circuit()
#     circuit_build_time = time.time() - start
#     tqdm.write(f"  Built in {circuit_build_time:.2f}s  |  Instructions : {len(circuit_init)}")

#     # Save
#     circuit_path = f"circuit_{tag}.pkl"
#     with open(circuit_path, "wb") as f:
#         pickle.dump({
#             "circuit_init":         circuit_init,
#             "num_physical_qubits":  num_physical_qubits,
#             "num_logical_qubits":   num_logical_qubits,
#             "num_total_qubits":     num_total_qubits,
#             "circuit_build_time":   circuit_build_time,
#             "n":                    n_code,
#             "k":                    k_code,
#             "d_est":                d_code,
#         }, f)
#     tqdm.write(f"  Saved : {circuit_path}")

# tqdm.write(f"\nPHASE 2 COMPLETE ✓ — {len(operator_files)} circuit files saved.\n")

In [10]:
tqdm.write("PHASE 3: Sampling over error rates\n")

# ── Error rate / shot configuration ──────────────────────────────────────────
num_error_levels = 20
error_rates      = np.logspace(0, -3, num_error_levels)   # 1.0 → 0.001 log-spaced
shot_counts      = [1000 * (i + 1) for i in range(num_error_levels)]  # 1000, 2000, ..., 20000
error_shots_dict = dict(zip(error_rates, shot_counts))

tqdm.write(f"Error rates (log-spaced) : {error_rates[0]:.4f} → {error_rates[-1]:.6f}")
tqdm.write(f"Shot counts (linear)     : {min(shot_counts):,} → {max(shot_counts):,}")

# ── Collect circuit files ─────────────────────────────────────────────────────
circuit_files = sorted(glob.glob("circuit_*.pkl"))
tqdm.write(f"\nFound {len(circuit_files)} circuit files.\n")

# ── Outer loop: codes ─────────────────────────────────────────────────────────
for code_idx, circ_path in enumerate(tqdm(circuit_files, desc="Codes", position=0, leave=True), start=1):

    base                   = os.path.basename(circ_path)
    _, n_str, k_str, d_str = base.replace(".pkl", "").split("_")
    n_code, k_code, d_code = int(n_str), int(k_str), int(d_str)
    tag                    = f"{n_code}_{k_code}_{d_code}"

    tqdm.write(f"\n{'='*60}")
    tqdm.write(f"[{code_idx}/{len(circuit_files)}] {base}")
    tqdm.write(f"  Parameters : [[n={n_code}, k={k_code}, d={d_code}]]")

    # ── Load circuit ──────────────────────────────────────────────────────────
    tqdm.write(f"  Loading circuit ...")
    with open(circ_path, "rb") as f:
        circ = pickle.load(f)

    circuit_init        = circ["circuit_init"]
    num_physical_qubits = circ["num_physical_qubits"]
    num_logical_qubits  = circ["num_logical_qubits"]
    num_total_qubits    = circ["num_total_qubits"]
    tqdm.write(f"  Physical : {num_physical_qubits}  |  Logical : {num_logical_qubits}  |  Total : {num_total_qubits}")

    # ── Inner loop: error rates ───────────────────────────────────────────────
    tqdm.write(f"  Starting sampling over {len(error_shots_dict)} error rates ...")
    results_dict       = {}
    total_shots_so_far = 0
    phase_start        = time.time()

    for depol_error_rate, num_shots in tqdm(
        error_shots_dict.items(),
        desc=f"  [[{n_code},{k_code},{d_code}]]",
        position=1,
        leave=False
    ):
        shot_start = time.time()

        # Build circuit for this error rate
        circuit = circuit_init.copy()
        circuit.append("DEPOLARIZE1", list(range(num_physical_qubits)), depol_error_rate)

        for qubit_index in range(num_total_qubits):
            target_qubit = qubit_index + num_total_qubits
            circuit.append("MXX", [qubit_index, target_qubit])
            circuit.append("MZZ", [qubit_index, target_qubit])

        # Sample
        tqdm.write(f"    Sampling : depol={depol_error_rate:.6f}  shots={num_shots:>8,} ...")
        sampler       = circuit.compile_sampler()
        measurements  = np.array(sampler.sample(shots=num_shots), dtype=np.uint8)
        shot_duration = time.time() - shot_start
        total_shots_so_far += num_shots

        results_dict[depol_error_rate] = {
            "num_shots":         num_shots,
            "measurements":      measurements,
            "sampling_time_sec": shot_duration,
        }
        tqdm.write(f"    Done     : depol={depol_error_rate:.6f}  shots={num_shots:>8,}  "
                   f"shape={measurements.shape}  "
                   f"time={shot_duration:.2f}s  "
                   f"cumulative shots={total_shots_so_far:,}")

    # ── Save samples ──────────────────────────────────────────────────────────
    sample_path = f"sample_{tag}.pkl"
    tqdm.write(f"\n  Saving samples to {sample_path} ...")
    with open(sample_path, "wb") as f:
        pickle.dump({
            "results":           results_dict,
            "error_rates":       list(error_shots_dict.keys()),
            "shot_counts":       list(error_shots_dict.values()),
            "n":                 n_code,
            "k":                 k_code,
            "d_est":             d_code,
            "total_shots":       total_shots_so_far,
            "total_time_sec":    time.time() - phase_start,
        }, f)

    tqdm.write(f"  Saved       : {sample_path}")
    tqdm.write(f"  Total shots : {total_shots_so_far:,}")
    tqdm.write(f"  Total time  : {time.time() - phase_start:.2f}s")
    tqdm.write(f"[{code_idx}/{len(circuit_files)}] COMPLETE ✓")

tqdm.write(f"\n{'='*60}")
tqdm.write(f"PHASE 3 COMPLETE ✓ — {len(circuit_files)} sample files saved.")
tqdm.write(f"{'='*60}")

PHASE 3: Sampling over error rates

Error rates (log-spaced) : 1.0000 → 0.001000
Shot counts (linear)     : 1,000 → 20,000

Found 10 circuit files.



Codes:   0%|          | 0/10 [00:00<?, ?it/s]


[1/10] circuit_1225_49_10.pkl
  Parameters : [[n=1225, k=49, d=10]]
  Loading circuit ...
  Physical : 1225  |  Logical : 49  |  Total : 1274
  Starting sampling over 20 error rates ...


                                             
Codes:   0%|          | 0/10 [00:01<?, ?it/s]           

    Sampling : depol=1.000000  shots=   1,000 ...


...

    Sampling : depol=0.001000  shots=  20,000 ...


                                                         
Codes:  90%|█████████ | 9/10 [2:11:59<18:16, 1096.92s/it]      

    Done     : depol=0.001000  shots=  20,000  shape=(20000, 1872)  time=2.22s  cumulative shots=210,000

  Saving samples to sample_900_36_8.pkl ...


Codes: 100%|██████████| 10/10 [2:12:00<00:00, 792.02s/it]

  Saved       : sample_900_36_8.pkl
  Total shots : 210,000
  Total time  : 44.09s
[10/10] COMPLETE ✓

PHASE 3 COMPLETE ✓ — 10 sample files saved.


In [16]:
from ldpc import BpOsdDecoder

def initialize_bp_osd_decoders(parity_check_x_matrix, parity_check_z_matrix, depol_error_rate):
    """Initializes BP-OSD decoders for X and Z errors."""
    bp_osd_x = BpOsdDecoder(
        parity_check_x_matrix,
        error_rate=float(depol_error_rate) * 2/3,
        bp_method='product_sum',
        max_iter=7,
        schedule='serial',
        osd_method='osd_cs',
        osd_order=2
    )

    bp_osd_z = BpOsdDecoder(
        parity_check_z_matrix,
        error_rate=float(depol_error_rate) * 2/3,
        bp_method='product_sum',
        max_iter=7,
        schedule='serial',
        osd_method='osd_cs',
        osd_order=2
    )
    
    return bp_osd_x, bp_osd_z


def partition_measurements(measurement_data, num_physical_qubits):
    """Splits measurement records into input and output phases for X and Z."""
    xx_input = measurement_data[::2][:num_physical_qubits]
    zz_input = measurement_data[1::2][:num_physical_qubits]
    xx_output = measurement_data[::2][num_physical_qubits:]
    zz_output = measurement_data[1::2][num_physical_qubits:]
    return xx_input, zz_input, xx_output, zz_output

def apply_error_corrections(x_input, z_input, x_output, z_output, decoded_x_errors, decoded_z_errors):
    """Applies corrections to logical X and Z phases."""
    logical_x_phase = logical_x_matrix @ x_input % 2
    logical_z_phase = logical_z_matrix @ z_input % 2

    logical_x_correction = logical_x_matrix @ decoded_z_errors % 2
    logical_z_correction = logical_z_matrix @ decoded_x_errors % 2

    final_x_phase = (x_output + logical_x_phase + logical_x_correction) % 2
    final_z_phase = (z_output + logical_z_phase + logical_z_correction) % 2
    return final_x_phase, final_z_phase

In [19]:
import numpy as np
import pickle
import glob
import os
import time
from tqdm import tqdm
from css import compute_css_logical_operators

tqdm.write("PHASE 4: Decoding and computing logical error rates\n")

# ── Collect operator and sample files ─────────────────────────────────────────
operator_files = sorted(glob.glob("/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/distillation/logical_ops/operators_*.pkl"))
tqdm.write(f"Found {len(operator_files)} operator files.")

# ── Outer loop: codes ─────────────────────────────────────────────────────────
for code_idx, op_path in enumerate(tqdm(operator_files, desc="Codes", position=0, leave=True), start=1):

    base                   = os.path.basename(op_path)
    _, n_str, k_str, d_str = base.replace(".pkl", "").split("_")
    n_code, k_code, d_code = int(n_str), int(k_str), int(d_str)
    tag                    = f"{n_code}_{k_code}_{d_code}"

    tqdm.write(f"\n{'='*60}")
    tqdm.write(f"[{code_idx}/{len(operator_files)}] [[n={n_code}, k={k_code}, d={d_code}]]")

    # ── Load operators from Phase 1 ───────────────────────────────────────────
    tqdm.write(f"  Loading operators from {op_path} ...")
    with open(op_path, "rb") as f:
        ops = pickle.load(f)

    parity_check_x_matrix = ops["hgp_x"].astype(np.uint8, copy=False)
    parity_check_z_matrix = ops["hgp_z"].astype(np.uint8, copy=False)
    logical_x_matrix       = ops["logical_x"]
    logical_z_matrix       = ops["logical_z"]
    num_physical_qubits    = ops["num_physical_qubits"]
    num_logical_qubits     = ops["num_logical_qubits"]
    num_total_qubits       = ops["num_total_qubits"]

    tqdm.write(f"  Physical : {num_physical_qubits}  |  Logical : {num_logical_qubits}  |  Total : {num_total_qubits}")

    # ── Load samples from Phase 3 ─────────────────────────────────────────────
    sample_path = f"/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/distillation/raw_samples/sample_{tag}.pkl"
    if not os.path.exists(sample_path):
        tqdm.write(f"  WARNING: Sample file {sample_path} not found, skipping.")
        continue

    tqdm.write(f"  Loading samples from {sample_path} ...")
    with open(sample_path, "rb") as f:
        sample_data = pickle.load(f)

    results      = sample_data["results"]
    tqdm.write(f"  Loaded {len(results)} error rate levels.")

    # ── Inner loop: error rates ───────────────────────────────────────────────
    tqdm.write(f"  Starting decoding over {len(results)} error rates ...")
    results_dict = {}
    phase_start  = time.time()

    for depol_error_rate, details in tqdm(
        results.items(),
        desc=f"  [[{n_code},{k_code},{d_code}]]",
        position=1,
        leave=False
    ):
        num_shots    = details["num_shots"]
        measurements = details["measurements"]

        tqdm.write(f"\n    depol={depol_error_rate:.6f}  shots={num_shots:,}")

        # Initialize decoders
        tqdm.write(f"    Initializing BP-OSD decoders ...")
        bp_osd_x, bp_osd_z = initialize_bp_osd_decoders(
            parity_check_x_matrix,
            parity_check_z_matrix,
            depol_error_rate
        )

        # Decode shot by shot
        num_errors  = 0
        shot_start  = time.time()

        for shot in tqdm(
            range(num_shots),
            desc=f"      shots @ depol={depol_error_rate:.4f}",
            position=2,
            leave=False
        ):
            xx_input, zz_input, xx_output, zz_output = partition_measurements(
                measurements[shot], num_physical_qubits
            )

            x_syndrome      = parity_check_x_matrix @ xx_input % 2
            decoded_z_errors = bp_osd_x.decode(x_syndrome)

            z_syndrome      = parity_check_z_matrix @ zz_input % 2
            decoded_x_errors = bp_osd_z.decode(z_syndrome)

            final_x_phase, final_z_phase = apply_error_corrections(
                xx_input, zz_input, xx_output, zz_output,
                decoded_x_errors, decoded_z_errors
            )

            num_errors += np.count_nonzero(np.bitwise_or(final_x_phase, final_z_phase))

        # Compute error rate and standard deviation
        output_error_rate = num_errors / (num_logical_qubits * num_shots)
        standard_deviation = np.sqrt(
            output_error_rate * (1 - output_error_rate) / num_shots
        )
        shot_duration = time.time() - shot_start

        tqdm.write(f"    Depolarization : {100 * depol_error_rate:.4f}%")
        tqdm.write(f"    Logical error  : {100 * output_error_rate:.4f} ± {100 * standard_deviation:.4f}%")
        tqdm.write(f"    Errors / shots : {num_errors} / {num_shots}  |  Time : {shot_duration:.2f}s")

        results_dict[depol_error_rate] = {
            "num_shots":          num_shots,
            "num_errors":         num_errors,
            "output_error_rate":  output_error_rate,
            "standard_deviation": standard_deviation,
            "decoding_time_sec":  shot_duration,
        }

    # ── Save results ──────────────────────────────────────────────────────────
    output_path = f"/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/distillation/processed_fid/fidelity_{tag}.pkl"
    tqdm.write(f"\n  Saving results to {output_path} ...")
    with open(output_path, "wb") as f:
        pickle.dump({
            "results":           results_dict,
            "n":                 n_code,
            "k":                 k_code,
            "d_est":             d_code,
            "total_time_sec":    time.time() - phase_start,
        }, f)

    tqdm.write(f"  Saved      : {output_path}")
    tqdm.write(f"  Total time : {time.time() - phase_start:.2f}s")
    tqdm.write(f"[{code_idx}/{len(operator_files)}] COMPLETE ✓")

tqdm.write(f"\n{'='*60}")
tqdm.write(f"PHASE 4 COMPLETE ✓ — {len(operator_files)} fidelity files saved.")
tqdm.write(f"{'='*60}")

PHASE 4: Decoding and computing logical error rates

Found 1 operator files.


Codes:   0%|          | 0/1 [00:00<?, ?it/s]


[1/1] [[n=400, k=16, d=6]]
  Loading operators from /Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/distillation/logical_ops/operators_400_16_6.pkl ...
  Physical : 400  |  Logical : 16  |  Total : 416
  Loading samples from /Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/distillation/raw_samples/sample_400_16_6.pkl ...
  Loaded 20 error rate levels.
  Starting decoding over 20 error rates ...



Codes:   0%|          | 0/1 [00:00<?, ?it/s]          


    depol=1.000000  shots=1,000
    Initializing BP-OSD decoders ...


...

    Depolarization : 48.3293%
    Logical error  : 75.0917 ± 0.7896%
    Errors / shots : 36044 / 3000  |  Time : 9.81s

    depol=0.335982  shots=4,000
    Initializing BP-OSD decoders ...









































































































































Codes:   0%|          | 0/1 [00:33<?, ?it/s]                  

    Depolarization : 33.5982%
    Logical error  : 74.7578 ± 0.6868%
    Errors / shots : 47845 / 4000  |  Time : 13.68s

    depol=0.233572  shots=5,000
    Initializing BP-OSD decoders ...






























Codes:   0%|          | 0/1 [00:35<?, ?it/s]


KeyboardInterrupt: 